In [1]:
# from 01-data.ipynb
import requests

URL_PREFIX='https://datatalks.club/faq'

path = '/json/courses.json'
response = requests.get(url=f'{URL_PREFIX}{path}')
courses_raw = response.json()

documents = []
for course in courses_raw:
    course_path = f'{URL_PREFIX}{course['path']}'
    course_response = requests.get(url=course_path)
    course_response.raise_for_status()
    course_faqs = course_response.json()

    documents.extend(course_faqs)

print(f'Total number of faq documents: {len(documents)}')

Total number of faq documents: 1346


In [2]:
# from 02-index-search.ipynb
from minsearch import Index

index = Index(
    text_fields=['question','section','answer'],
    keyword_fields=['course']
)

index.fit(documents)

def search(question, course='llm-zoomcamp'):
    boost_dict = {'question':2.0, 'section':0.5}
    filter_dict={'course':course}

    return index.search(
        query=question,
        filter_dict=filter_dict,
        boost_dict=boost_dict,
        num_results=5
    )

In [3]:
SYSTEM_PROMPT = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

USER_PROMPT_TEMPLATE = '''
Question:
{question}

Context:
{context}
'''

In [4]:
# format (preprocess) the search results into a context input that is easily understood by the llm
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc['section'])
        lines.append(f'Q: {doc['question']}')
        lines.append(f'A: {doc['answer']}')
        lines.append('')
    
    return '\n'.join(lines).strip()

In [5]:
# build the user prompt using prompt template, search results and user query
def build_prompt(question, search_results):
    context = build_context(search_results=search_results)
    prompt =  USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )

    return prompt.strip()

In [6]:
question = 'I just discovered the course. Can I join now?'
search_results = search(question)
prompt = build_prompt(question=question, search_results=search_results)
print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project